# V7.2 — Adaptive Tiling (PARSER FIXED / CACHE SAFE)
# V7.2 — Targeted Adaptive Tiling + Visual Verification (FSC-147)

**Mục đích:** thử cách cứu các câu truy vấn bị CountGD++ trả **0 detection**. Sử dụng bộ V7 gốc đã có, chỉ chạy CountGD++ trên các vùng của ảnh thuộc nhóm này; dùng Qwen3-VL để kiểm tra box mới. **Không đưa `gt_count`, `pair_type`, `source_class` hoặc nhãn test vào bất cứ prompt, logic chọn ảnh hay quy tắc dự đoán nào.**

Đầu vào trên Kaggle: dataset ảnh FSC147 (`hihihi5`) và ZIP `V7_1_EXPLORATORY_eval_1301.zip` (chứa `V7_1_EXPLORATORY_1301_predictions.csv`, cả V7 và CountGD baseline).

**Giới hạn khoa học:** đây vẫn là thí nghiệm khám phá trên TEST đã được xem trong nghiên cứu. Không được gọi kết quả này là bằng chứng tổng quát hóa. Muốn paper có giá trị, sau khi chốt V7.2 phải kiểm tra tập mới, đặc biệt hard-negative với CountGD baseline = 0.

**Chạy 1 lần:** bật GPU và Internet rồi `Run All`. Notebook tự checkpoint theo ảnh; khi bị Kaggle ngắt có thể Add Input file checkpoint và chạy lại. Model Qwen kiểm tra mục tiêu của crop/ảnh, nhưng **cùng họ model với V4**, không phải một verifier độc lập hoàn toàn.

In [ ]:
# 01 — CONFIG (đặt trước khi chạy)
import os, sys, json, time, re, zipfile, gc, math, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

SEED = 42
np.random.seed(SEED)
ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
OUT = ROOT / 'v72_adaptive_tiling'
OUT.mkdir(parents=True, exist_ok=True)
TILE_CHECKPOINT = OUT / 'v72_tiles_checkpoint.csv'
GATE_CHECKPOINT = OUT / 'v72_visual_gate_checkpoint.csv'
BOX_STORE = OUT / 'v72_tile_boxes'
BOX_STORE.mkdir(exist_ok=True)

# Freeze before comparing against TEST results
COUNTGD_THRESH = 0.23
TILE_OVERLAP_FRAC = 0.20   # crop mở rộng so với phân vùng lõi
NMS_IOU = 0.50
COARSE_GRID = 2
FINE_GRID = 4
ESCALATE_IF_COUNT_BELOW = 250  # chỉ quan sát đầu ra CountGD, KHÔNG dùng GT
QWEN_SAMPLE_MAX = 6
QWEN_MIN_YES = 4
QWEN_MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
COUNTGD_GITHUB = 'https://github.com/niki-amini-naieni/CountGDPlusPlus.git'
COUNTGD_REPO = ROOT / 'CountGDPlusPlus'
COUNTGD_GDRIVE_ID = '1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs'
MAX_IMAGES = None  # None => chạy tất cả trường hợp đã kích hoạt; có thể đặt 2 để smoke

print('Output:',OUT)


In [ ]:
# 02 — ĐỌC DỰ ĐOÁN CŨ / TÌM ẢNH (không phụ thuộc tên Kaggle Input)
# Kaggle thường tự giải nén ZIP thành các file CSV riêng trong Input.
# Ưu tiên đọc CSV trực tiếp; chỉ mở ZIP nếu thực sự tìm thấy ZIP.
import io

search_roots = [Path('/kaggle/input'), Path('/mnt/data')]
search_roots = [r for r in search_roots if r.exists()]
prediction_name = 'V7_1_EXPLORATORY_1301_predictions.csv'
zip_name = 'V7_1_EXPLORATORY_eval_1301.zip'

csv_candidates = []
for root_path in search_roots:
    csv_candidates.extend(root_path.rglob(prediction_name))

if csv_candidates:
    source_path = sorted(csv_candidates, key=str)[0]
    data = pd.read_csv(source_path)
    print('Đã đọc CSV trực tiếp (Kaggle đã giải nén ZIP):', source_path)
else:
    zip_candidates = []
    for root_path in search_roots:
        zip_candidates.extend(root_path.rglob(zip_name))
    if not zip_candidates:
        raise FileNotFoundError(
            'Không tìm thấy '+prediction_name+' hoặc '+zip_name+
            ' trong Kaggle Input. Hãy Add Input dataset V7_1_EXPLORATORY_eval_1301.'
        )
    source_path = sorted(zip_candidates, key=str)[0]
    with zipfile.ZipFile(source_path) as z:
        matches = [name for name in z.namelist() if Path(name).name == prediction_name]
        if not matches:
            raise FileNotFoundError(f'ZIP không chứa {prediction_name}')
        data = pd.read_csv(z.open(matches[0]))
    print('Đã đọc CSV từ ZIP:', source_path)

required = {'group_id','image_name','target','baseline_count','v4_filtered_count','v7_count','gt_count','pair_type'}
assert required.issubset(data.columns), sorted(required-set(data.columns))
assert data.group_id.is_unique and len(data)==1301

# ĐIỀU KIỆN CHỌN ẢNH KHÔNG DÙNG GROUND TRUTH:
triggers = data[(data.baseline_count==0)&(data.v7_count==0)].copy()
triggers = triggers.sort_values('image_name').reset_index(drop=True)
if MAX_IMAGES is not None:
    triggers=triggers.head(MAX_IMAGES).copy()
print('Tổng cặp:',len(data),'| Số cặp cần tiling:',len(triggers))
assert len(triggers)>0, 'Không có trường hợp kích hoạt tiling'

# Index ảnh chỉ một lần
root = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/mnt/data')
image_index={}
for path in root.rglob('*.jpg'):
    if path.name not in image_index:
        image_index[path.name]=str(path)
missing=sorted(set(triggers.image_name)-set(image_index))
if missing:
    raise FileNotFoundError(f'Thiếu {len(missing)} ảnh trong Kaggle Input, ví dụ: {missing[:6]}')
triggers['image_path']=triggers.image_name.map(image_index)
print('Ảnh đã định vị:',len(triggers),'| mô hình không nhìn thấy GT')
print(triggers[['group_id','image_name','target','baseline_count','v7_count']].head(8).to_string(index=False))


In [ ]:
# 02b — RESTORE CHECKPOINT từ ZIP ở Kaggle Input (nếu đã chạy một phần)
import shutil, zipfile
old_zips=list(Path('/kaggle/input').rglob('V7_2_partial_checkpoint.zip')) if Path('/kaggle/input').exists() else []
old_zips+=list(Path('/kaggle/input').rglob('V7_2_Adaptive_Tiling_Exploratory_1301.zip')) if Path('/kaggle/input').exists() else []
if old_zips:
    zpath=old_zips[-1]
    with zipfile.ZipFile(zpath) as z:
        for member in z.namelist():
            if member in ('v72_tiles_checkpoint.csv','v72_visual_gate_checkpoint.csv'):
                dst=OUT/member
            elif member.startswith('tile_boxes/') and member.endswith('.json') and '..' not in member:
                dst=BOX_STORE/Path(member).name
            else:
                continue
            if not dst.exists():
                dst.parent.mkdir(parents=True,exist_ok=True)
                dst.write_bytes(z.read(member))
    print('Restored from',zpath)
else:
    print('Không có checkpoint cũ; bắt đầu mới')

def snapshot_partial():
    path=ROOT/'V7_2_partial_checkpoint.zip'
    with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
        for file in (TILE_CHECKPOINT,GATE_CHECKPOINT):
            if file.exists():z.write(file,arcname=file.name)
        for file in BOX_STORE.glob('*.json'):
            z.write(file,arcname='tile_boxes/'+file.name)
    return path


## A — Chạy CountGD++ trên các ô ảnh

Notebook tái sử dụng phần cài môi trường/loader đã chạy thành công của notebook V7 trước đó. Mỗi ảnh chạy 2×2 ô; nếu số detection thấp hơn ngưỡng cố định, chạy tiếp 4×4 ô. Các ô có vùng overlap để tránh bỏ sót vật thể ở rìa; một detection chỉ được tính trong vùng lõi tương ứng, sau đó NMS toàn ảnh để khử trùng.

Lưu checkpoint **sau mỗi ảnh**. Chạy lại sẽ bỏ qua ảnh đã xong.

In [ ]:
# ============================================================
# CELL 2 — DEPENDENCIES + COUNTGD++ REPO
# Giữ Torch/Torchvision hiện tại.
# ============================================================

import sys
import subprocess
import importlib.metadata as im
from pathlib import Path

def pkg_version(name):
    try:
        return im.version(name)
    except Exception:
        return None

print("Existing torch:", pkg_version("torch"))
print("Existing torchvision:", pkg_version("torchvision"))
print("Existing transformers:", pkg_version("transformers"))

if not COUNTGD_REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", COUNTGD_GITHUB, str(COUNTGD_REPO)],
        check=True,
    )

req_file = COUNTGD_REPO / "requirements.txt"
if req_file.exists():
    blocked = (
        "torch", "torchvision", "torchaudio", "numpy", "scipy", "opencv",
        "gradio", "spaces", "deeplake"
    )
    keep = []
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(s.lower().startswith(x) for x in blocked):
            continue
        keep.append(s)

    filtered = Path("/kaggle/working/countgd_v4_requirements_filtered.txt")
    filtered.write_text("\n".join(keep) + "\n", encoding="utf-8")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(filtered)],
        check=True,
    )

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "transformers>=4.57.0,<5",
        "accelerate>=1.2.0",
        "sentencepiece",
        "gdown",
        "qwen-vl-utils",
    ],
    check=True,
)

print("Dependency setup finished.")
print("transformers:", pkg_version("transformers"))


In [ ]:
# ============================================================
# CELL 3 — VERIFY TORCH/CUDA + PATCH/BUILD GROUNDINGDINO OPS
# ============================================================

import os
import re
import sys
import gc
import importlib
import subprocess
from pathlib import Path

import torch
import torchvision

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if not torch.cuda.is_available():
    raise RuntimeError("V3 cần GPU Kaggle.")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

from torchvision.ops import nms
_test_boxes = torch.tensor([[0.,0.,10.,10.],[1.,1.,9.,9.]], device="cuda")
_test_scores = torch.tensor([0.9,0.8], device="cuda")
_ = nms(_test_boxes, _test_scores, 0.5)
print("torchvision CUDA NMS: PASS")

def detectron2_ok():
    try:
        import detectron2
        from detectron2 import _C
        return True
    except Exception as e:
        print("Detectron2 check:", repr(e))
        return False

if not detectron2_ok():
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation",
            "git+https://github.com/facebookresearch/detectron2.git",
        ],
        check=True,
    )
    importlib.invalidate_caches()
    if not detectron2_ok():
        raise RuntimeError("Detectron2 vẫn không load được sau khi cài.")

OPS_DIR = COUNTGD_REPO / "models" / "GroundingDINO" / "ops"
SRC_DIR = OPS_DIR / "src"
assert OPS_DIR.exists(), OPS_DIR

source_suffixes = {".h", ".hpp", ".cpp", ".cu", ".cuh"}
patched = []
for path in SRC_DIR.rglob("*"):
    if not path.is_file() or path.suffix not in source_suffixes:
        continue
    original = path.read_text(encoding="utf-8", errors="ignore")
    updated = original
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.is_cuda\(\)",
        r"\1.is_cuda()",
        updated,
    )
    updated = re.sub(
        r"([A-Za-z_][A-Za-z0-9_]*)\.type\(\)\.scalar_type\(\)",
        r"\1.scalar_type()",
        updated,
    )
    updated = updated.replace("value.type()", "value.scalar_type()")
    if updated != original:
        path.write_text(updated, encoding="utf-8")
        patched.append(str(path.relative_to(OPS_DIR)))

print("Patched files:", patched if patched else "already compatible")

from torch.utils.cpp_extension import CUDA_HOME
if CUDA_HOME:
    os.environ["CUDA_HOME"] = str(CUDA_HOME)

major, minor = torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = f"{major}.{minor}"
os.environ["MAX_JOBS"] = "1"

if Path("/usr/bin/gcc-11").exists():
    os.environ["CC"] = "/usr/bin/gcc-11"
if Path("/usr/bin/g++-11").exists():
    os.environ["CXX"] = "/usr/bin/g++-11"

def try_import_msda():
    try:
        if str(OPS_DIR) not in sys.path:
            sys.path.insert(0, str(OPS_DIR))
        importlib.invalidate_caches()
        return importlib.import_module("MultiScaleDeformableAttention")
    except Exception:
        return None

MSDA = try_import_msda()
if MSDA is None:
    print("Building GroundingDINO CUDA ops...")
    subprocess.run(
        [sys.executable, "setup.py", "build_ext", "--inplace"],
        cwd=str(OPS_DIR),
        check=True,
    )
    importlib.invalidate_caches()
    sys.modules.pop("MultiScaleDeformableAttention", None)
    MSDA = try_import_msda()

if MSDA is None:
    raise RuntimeError("Không import được MultiScaleDeformableAttention sau build.")

print("GroundingDINO CUDA ops: PASS", MSDA.__file__)


In [ ]:
# ============================================================
# CELL 4 — DOWNLOAD COUNTGD++ WEIGHTS
# ============================================================

import sys
import subprocess
from pathlib import Path

CHECKPOINT_DIR = COUNTGD_REPO / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BERT_DIR = CHECKPOINT_DIR / "bert-base-uncased"
if not (BERT_DIR / "config.json").exists():
    print("Downloading bert-base-uncased...")
    subprocess.run(
        [sys.executable, "download_bert.py"],
        cwd=str(COUNTGD_REPO),
        check=True,
    )

COUNTGD_CKPT = CHECKPOINT_DIR / "countgd_plusplus.pth"
if (not COUNTGD_CKPT.exists()) or COUNTGD_CKPT.stat().st_size < 500 * 1024**2:
    print("Downloading CountGD++ checkpoint...")
    subprocess.run(
        [sys.executable, "-m", "gdown", COUNTGD_GDRIVE_ID, "-O", str(COUNTGD_CKPT)],
        check=True,
    )

assert (BERT_DIR / "config.json").exists()
assert COUNTGD_CKPT.exists()
print("BERT:", BERT_DIR)
print("CountGD++ checkpoint GB:", round(COUNTGD_CKPT.stat().st_size / 1024**3, 2))


In [ ]:
# 06 — LOAD COUNTGD++ (giữ nguyên API cũ)
import torch, importlib
import gc
p=str(COUNTGD_REPO)
if p not in sys.path:sys.path.insert(0,p)
os.chdir(str(COUNTGD_REPO))
for name in list(sys.modules):
    if name=='models' or name.startswith('models.'):
        del sys.modules[name]
importlib.invalidate_caches();gc.collect();torch.cuda.empty_cache()
process_file=COUNTGD_REPO/'process_folder.py'
source=process_file.read_text(encoding='utf-8')
marker='logging.basicConfig(level=logging.INFO'
if marker not in source:raise RuntimeError('API CountGD++ thay đổi')
countgd_namespace={'__name__':'v72_countgd_runtime','__file__':str(process_file)}
exec(compile(source.split(marker,1)[0],str(process_file),'exec'),countgd_namespace)
for k in ['build_arg_parser','build_model_and_transforms','count']:
    if k not in countgd_namespace:raise RuntimeError(f'Missing CountGD++ API: {k}')
args=countgd_namespace['build_arg_parser']().parse_args([])
args.device='cuda:0';args.pretrain_model_path=str(COUNTGD_CKPT)
_load=torch.load
def _load_trusted(*a,**kw):
    kw.setdefault('weights_only',False)
    return _load(*a,**kw)
torch.load=_load_trusted
try:
    countgd_model,countgd_transform=countgd_namespace['build_model_and_transforms'](args)
finally:
    torch.load=_load
countgd_model=countgd_model.to('cuda:0').eval()
countgd_namespace.update(model=countgd_model,transform=countgd_transform,device=torch.device('cuda:0'))
print('CountGD++ loaded')


In [ ]:
# 07 — HELPERS: TILE, TỌA ĐỘ TOÀN ẢNH, NMS, CHECKPOINT

def atomic_csv(df,path):
    temp=Path(str(path)+'.tmp')
    df.to_csv(temp,index=False)
    temp.replace(path)

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    inter=max(0,min(ax2,bx2)-max(ax1,bx1))*max(0,min(ay2,by2)-max(ay1,by1))
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1)
    area_b=max(0,bx2-bx1)*max(0,by2-by1)
    return inter/max(1e-9,area_a+area_b-inter)

def nms_rows(rows,thr=NMS_IOU):
    chosen=[]
    for row in sorted(rows,key=lambda r:-float(r['score'])):
        if not any(iou(row['box'],prev['box'])>thr for prev in chosen):
            chosen.append(row)
    return chosen

def count_tiled(img,target,grid):
    W,H=img.size
    detections=[]
    for gy in range(grid):
        for gx in range(grid):
            core_x0=int(round(gx*W/grid));core_x1=int(round((gx+1)*W/grid))
            core_y0=int(round(gy*H/grid));core_y1=int(round((gy+1)*H/grid))
            dx=(core_x1-core_x0)*TILE_OVERLAP_FRAC
            dy=(core_y1-core_y0)*TILE_OVERLAP_FRAC
            x0=max(0,int(math.floor(core_x0-dx)))
            x1=min(W,int(math.ceil(core_x1+dx)))
            y0=max(0,int(math.floor(core_y0-dy)))
            y1=min(H,int(math.ceil(core_y1+dy)))
            crop=img.crop((x0,y0,x1,y1))
            if min(crop.size)<8:continue
            boxes,scores,_=countgd_namespace['count'](crop,str(target).replace('.','').strip(),{'image':crop,'points':[]},[],[],float(COUNTGD_THRESH),False)
            boxes=boxes.tolist() if hasattr(boxes,'tolist') else list(boxes)
            scores=scores.tolist() if hasattr(scores,'tolist') else list(scores)
            for box,score in zip(boxes,scores):
                score=float(score)
                if score<=COUNTGD_THRESH:continue
                xx0,yy0,xx1,yy1=[float(v) for v in box[:4]]
                # CountGD++ gốc trả bbox xyxy theo pixel của PIL image.
                xx0=min(max(xx0+x0,0),W);xx1=min(max(xx1+x0,0),W)
                yy0=min(max(yy0+y0,0),H);yy1=min(max(yy1+y0,0),H)
                bx0,bx1=sorted([xx0,xx1]);by0,by1=sorted([yy0,yy1])
                cx=(bx0+bx1)/2;cy=(by0+by1)/2
                # Ownership by center, tránh đếm hai lần ở các ô overlap.
                inside_x = core_x0<=cx<(core_x1 if gx<grid-1 else core_x1+1)
                inside_y = core_y0<=cy<(core_y1 if gy<grid-1 else core_y1+1)
                if inside_x and inside_y and bx1>bx0 and by1>by0:
                    detections.append({'box':[bx0,by0,bx1,by1],'score':score,'grid':grid,'cell':[gx,gy]})
    return nms_rows(detections)

# small self-test of NMS and IoU
assert iou([0,0,10,10],[0,0,10,10])>0.999
assert len(nms_rows([{'box':[0,0,10,10],'score':.9},{'box':[1,1,10,10],'score':.8}]))==1
print('Helpers ready; NMS self-test PASS')


In [ ]:
# 08 — TILING REAL INFERENCE + RESUME
if not torch.cuda.is_available():raise RuntimeError('Bật GPU Kaggle trước khi chạy!')
if not TILE_CHECKPOINT.exists():
    # Input checkpoint từ lần chạy cũ, nếu đã Add Input
    oldfiles=list(Path('/kaggle/input').rglob('v72_tiles_checkpoint.csv')) if Path('/kaggle/input').exists() else []
    if oldfiles:
        import shutil
        shutil.copy2(oldfiles[-1],TILE_CHECKPOINT)
        print('Đã khôi phục checkpoint:',oldfiles[-1])

rows=[]
if TILE_CHECKPOINT.exists(): rows=pd.read_csv(TILE_CHECKPOINT).to_dict('records')
done={str(x['group_id']) for x in rows if x.get('status')=='ok' and (BOX_STORE/(str(x['group_id']).replace('/','_')+'.json')).exists()}
for i,r in enumerate(triggers.itertuples(index=False),1):
    gid=str(r.group_id)
    if gid in done:
        print(f'[SKIP] {i}/{len(triggers)} {gid}')
        continue
    print(f'[{i}/{len(triggers)}] {gid}: {r.target}',flush=True)
    t0=time.time()
    try:
        img=Image.open(r.image_path).convert('RGB')
        coarse=count_tiled(img,r.target,COARSE_GRID)
        fine=[]
        if len(coarse)<ESCALATE_IF_COUNT_BELOW:
            fine=count_tiled(img,r.target,FINE_GRID)
            selected=fine; selected_grid=FINE_GRID
        else:
            selected=coarse; selected_grid=COARSE_GRID
        boxfile=BOX_STORE/(gid.replace('/','_')+'.json')
        boxfile.write_text(json.dumps(selected,ensure_ascii=False),encoding='utf-8')
        new={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'grid2_count':len(coarse),'grid4_count':len(fine) if selected_grid==FINE_GRID else -1,
             'selected_grid':selected_grid,'tile_count':len(selected),
             'latency_sec':round(time.time()-t0,2),'status':'ok','error':''}
    except Exception as e:
        new={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'grid2_count':None,'grid4_count':None,'selected_grid':None,'tile_count':None,
             'latency_sec':round(time.time()-t0,2),'status':'error','error':repr(e)}
        print('ERROR',repr(e),flush=True)
    rows=[x for x in rows if str(x['group_id'])!=gid]+[new]
    atomic_csv(pd.DataFrame(rows),TILE_CHECKPOINT)
    print('Checkpoint ZIP:',snapshot_partial())
    if new['status']!='ok':
        raise RuntimeError(f'Inference lỗi ở {gid}; giữ checkpoint để retry')
    print('grid2:',new['grid2_count'],'grid selected:',new['selected_grid'],'tile count:',new['tile_count'])
print('Stage TILING completed:',len(rows),'| checkpoint:',TILE_CHECKPOINT)


In [ ]:
# 09 — GIẢI PHÓNG COUNTGD TRƯỚC KHI LOAD QWEN
import torch,gc
for name in ['countgd_model','countgd_transform']:
    if name in globals():del globals()[name]
if 'countgd_namespace' in globals():
    countgd_namespace.clear()
    del countgd_namespace
gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print('Đã dọn VRAM CountGD')


## B — Qwen xác nhận các bounding box mới

Chỉ cho phép tăng dự đoán nếu có ít nhất **4/6** crop đại diện được Qwen xác nhận đúng target **và** kiểm tra ảnh toàn cảnh trả YES. Khi số candidate nhỏ hơn 4, giữ V7 gốc (fail-closed). Việc này giúp tránh dùng CountGD tiling như một phép khẳng định target tồn tại khi chưa được kiểm chứng.

**Không đưa số lượng thật vào prompt**. Qwen chỉ được thấy ảnh, crops và tên target. Khi Qwen sai vẫn có thể phát sinh FP; cần đánh giá trên hard-negative mới.

In [ ]:
# 10 — LOAD QWEN3-VL-4B
import torch,gc
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
qwen_processor=AutoProcessor.from_pretrained(QWEN_MODEL_ID)
qwen_model=Qwen3VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,dtype=torch.float16,device_map='auto',low_cpu_mem_usage=True).eval()
PANEL_SIZE=224
CROP_PADDING_FRAC=.30
print('Qwen loaded:',QWEN_MODEL_ID)


In [ ]:
# ============================================================
# CELL 11 — CROP / CONTACT-SHEET / QWEN VERIFIER UTILITIES
# ============================================================

from PIL import Image, ImageDraw, ImageFont, ImageOps
import pandas as pd
import numpy as np
import json
import re
import math
import torch

def expanded_crop(image, box, padding_frac=CROP_PADDING_FRAC):
    W,H = image.size
    x1,y1,x2,y2 = [float(v) for v in box]
    bw = max(1.0, x2-x1)
    bh = max(1.0, y2-y1)
    px = bw * padding_frac
    py = bh * padding_frac
    xx1 = int(max(0, math.floor(x1-px)))
    yy1 = int(max(0, math.floor(y1-py)))
    xx2 = int(min(W, math.ceil(x2+px)))
    yy2 = int(min(H, math.ceil(y2+py)))
    crop = image.crop((xx1,yy1,xx2,yy2)).convert("RGB")
    return crop

def make_contact_sheet(image, rows):
    n = len(rows)
    cols = 4
    rows_n = int(math.ceil(n / cols))
    label_h = 28
    sheet = Image.new(
        "RGB",
        (cols * PANEL_SIZE, rows_n * (PANEL_SIZE + label_h)),
        "white",
    )
    draw = ImageDraw.Draw(sheet)

    for local_id, row in enumerate(rows):
        box = json.loads(row["box_json"])
        crop = expanded_crop(image, box)
        crop = ImageOps.contain(crop, (PANEL_SIZE, PANEL_SIZE))

        x0 = (local_id % cols) * PANEL_SIZE
        y0 = (local_id // cols) * (PANEL_SIZE + label_h)

        panel = Image.new("RGB", (PANEL_SIZE, PANEL_SIZE), "white")
        px = (PANEL_SIZE - crop.width) // 2
        py = (PANEL_SIZE - crop.height) // 2
        panel.paste(crop, (px,py))
        sheet.paste(panel, (x0,y0))

        draw.rectangle(
            [x0, y0, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            outline="black",
            width=2,
        )
        draw.rectangle(
            [x0, y0+PANEL_SIZE, x0+PANEL_SIZE-1, y0+PANEL_SIZE+label_h-1],
            fill="white",
        )
        draw.text(
            (x0+6, y0+PANEL_SIZE+5),
            f"ID {local_id}",
            fill="black",
        )

    return sheet

LINE_RE = re.compile(r"^\s*(?:ID\s*)?(\d+)\s*\|\s*(YES|NO)\s*$", re.I)

def parse_sheet_output(text, n):
    parsed = {}
    for line in str(text).splitlines():
        m = LINE_RE.match(line.strip())
        if m:
            idx = int(m.group(1))
            if 0 <= idx < n:
                parsed[idx] = m.group(2).upper()
    return parsed

@torch.inference_mode()
def qwen_verify_sheet(image, target, sheet_rows):
    sheet = make_contact_sheet(image, sheet_rows)

    prompt = f"""
The second image is a numbered contact sheet of candidate detections from an object-counting model.

Target category: {target}

Task:
For EACH numbered panel, decide whether the main object shown in that panel is truly an instance of the target category "{target}".

Important:
- Judge the object in the numbered crop, not merely whether the target exists somewhere in the whole scene.
- Similar-looking or semantically related objects are NOT the target.
- Be conservative about rejecting a real target.
- Output exactly one line per panel and nothing else.
- Format exactly: 0|YES or 0|NO (use the numeric panel id; do not write the word ID)

Example:
0|YES
1|NO
2|YES
""".strip()

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "image", "image": sheet},
            {"type": "text", "text": prompt},
        ],
    }]

    text_prompt = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        messages,
        image_patch_size=16,
        return_video_kwargs=True,
        return_video_metadata=True,
    )

    video_metadatas = None
    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)

    inputs = qwen_processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        padding=True,
        return_tensors="pt",
        do_resize=False,
        **video_kwargs,
    )
    inputs = inputs.to(next(qwen_model.parameters()).device)

    generated_ids = qwen_model.generate(
        **inputs,
        max_new_tokens=max(64, 10 * len(sheet_rows) + 32),
        do_sample=False,
    )

    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text, sheet

print("Verifier utilities ready.")


In [ ]:
# 12 — KIỂM TRA ẢNH TOÀN CẢNH (1 câu hỏi khác với xác minh từng box)
@torch.inference_mode()
def qwen_full_presence(image,target):
    prompt=(f'Do you see any actual instance of the target category "{target}" in this image? '
            'Avoid mistaking similar-looking, related, or background objects for the target. '
            'Answer exactly YES, NO, or UNCERTAIN. Do not infer presence from the wording of the question.')
    messages=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    text_prompt=qwen_processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    im_in,vid_in,video_kwargs=process_vision_info(messages,image_patch_size=16,return_video_kwargs=True,return_video_metadata=True)
    video_metadatas=None
    if vid_in is not None:
        vid_in,video_metadatas=zip(*vid_in)
        vid_in=list(vid_in);video_metadatas=list(video_metadatas)
    inp=qwen_processor(text=[text_prompt],images=im_in,videos=vid_in,video_metadata=video_metadatas,
                       padding=True,return_tensors='pt',do_resize=False,**video_kwargs)
    inp=inp.to(next(qwen_model.parameters()).device)
    generated=qwen_model.generate(**inp,max_new_tokens=16,do_sample=False)
    raw=qwen_processor.batch_decode([out[len(ids):] for out,ids in zip(generated,inp.input_ids)],
                                     skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
    token=re.findall(r'\b(?:YES|NO|UNCERTAIN)\b',raw.upper())
    return (token[0] if len(token)==1 else 'UNCERTAIN'),raw

def representative_rows(boxes,topk=QWEN_SAMPLE_MAX):
    # Highest confidence, đa dạng vị trí (không chọn 6 boxes cùng vùng)
    selected=[];used=set()
    for row in sorted(boxes,key=lambda r:-float(r['score'])):
        cell=tuple(row.get('cell',[0,0]))
        if cell in used:continue
        selected.append(row);used.add(cell)
        if len(selected)>=topk:return [{'box_json':json.dumps(x['box'])} for x in selected]
    for row in sorted(boxes,key=lambda r:-float(r['score'])):
        if row not in selected:selected.append(row)
        if len(selected)>=topk:break
    return [{'box_json':json.dumps(x['box'])} for x in selected]


In [ ]:
# 13 — QWEN GATE + RESUME
if not GATE_CHECKPOINT.exists():
    olds=list(Path('/kaggle/input').rglob('v72_visual_gate_checkpoint.csv')) if Path('/kaggle/input').exists() else []
    if olds:
        import shutil
        shutil.copy2(olds[-1],GATE_CHECKPOINT)
        print('Đã khôi phục Qwen checkpoint:',olds[-1])
rows=[]
if GATE_CHECKPOINT.exists():rows=pd.read_csv(GATE_CHECKPOINT).to_dict('records')
done={str(x['group_id']) for x in rows if x.get('status')=='ok'}
for i,r in enumerate(triggers.itertuples(index=False),1):
    gid=str(r.group_id)
    if gid in done:continue
    tiling_row=pd.read_csv(TILE_CHECKPOINT).set_index('group_id').loc[gid]
    if tiling_row.status!='ok':raise RuntimeError('Tiling incomplete: '+gid)
    boxes=json.loads((BOX_STORE/(gid.replace('/','_')+'.json')).read_text(encoding='utf-8'))
    t0=time.time()
    try:
        if len(boxes)<QWEN_MIN_YES:
            yes=0; n_checked=0; full='UNCERTAIN';raw_full='';raw_crops=''
            gate='NO_SUFFICIENT_DETECTIONS'
        else:
            img=Image.open(r.image_path).convert('RGB')
            full,raw_full=qwen_full_presence(img,r.target)
            representative=representative_rows(boxes)
            raw_crops,_=qwen_verify_sheet(img,r.target,representative)
            parsed=parse_sheet_output(raw_crops,len(representative))
            # Không dùng fallback YES khi parsing lỗi
            n_checked=len(representative)
            yes=sum(1 for v in parsed.values() if v=='YES')
            complete=len(parsed)==n_checked
            gate=('CONFIRMED' if full=='YES' and complete and yes>=QWEN_MIN_YES
                  else 'NOT_CONFIRMED')
        # Gate chỉ cho thay dự đoán nếu Qwen xác nhận đủ mạnh
        new_count=int(len(boxes)) if gate=='CONFIRMED' else int(r.v7_count)
        record={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'baseline_count':int(r.baseline_count),'v7_count':int(r.v7_count),
             'tile_count':len(boxes),'full_presence':full,'n_crops':n_checked,'crop_yes':yes,
             'gate':gate,'v72_count':new_count,'latency_sec':round(time.time()-t0,2),
             'raw_full':raw_full,'raw_crops':raw_crops,'status':'ok','error':''}
    except Exception as e:
        record={'group_id':gid,'image_name':r.image_name,'target':r.target,
             'baseline_count':int(r.baseline_count),'v7_count':int(r.v7_count),
             'tile_count':len(boxes),'full_presence':'ERROR','n_crops':0,'crop_yes':0,
             'gate':'ERROR','v72_count':int(r.v7_count),'latency_sec':round(time.time()-t0,2),
             'raw_full':'','raw_crops':'','status':'error','error':repr(e)}
    rows=[x for x in rows if str(x['group_id'])!=gid]+[record]
    atomic_csv(pd.DataFrame(rows),GATE_CHECKPOINT)
    print('Checkpoint ZIP:',snapshot_partial())
    print(f'[{i}/{len(triggers)}] {gid}: tile={len(boxes)} gate={record["gate"]} -> {record["v72_count"]}')
    if record['status']!='ok':raise RuntimeError('Qwen gate failed: '+gid)
print('Qwen gate done:',len(rows))


In [ ]:
# 13b — SỬA GATE BỊ CACHE SAI DO LỖI REGEX (NO GPU, NO NEW QWEN CALLS)
# Chạy SAU Cell 13 và TRƯỚC Cell 14. Tái đọc raw_full và raw_crops đã lưu.
# Đây là sửa lỗi đọc đầu ra, KHÔNG thay đổi threshold quyết định.
import re
full_rx = re.compile(r'\b(?:YES|NO|UNCERTAIN)\b',re.I)
if GATE_CHECKPOINT.exists():
    cached=pd.read_csv(GATE_CHECKPOINT)
    repaired=0
    for idx, old in cached.iterrows():
        if str(old.get('status',''))!='ok':continue
        raw='' if pd.isna(old.get('raw_full')) else str(old.get('raw_full'))
        tokens=full_rx.findall(raw.upper())
        full=tokens[0].upper() if len(tokens)==1 else 'UNCERTAIN'
        n=int(old['n_crops'])
        raw_crops=old.get('raw_crops')
        parsed=parse_sheet_output('' if pd.isna(raw_crops) else str(raw_crops),n) if n>0 else {}
        yes=sum(v=='YES' for v in parsed.values())
        confirmed=(int(old['tile_count'])>=QWEN_MIN_YES and full=='YES' and
                   n>=QWEN_MIN_YES and len(parsed)==n and yes>=QWEN_MIN_YES)
        new_count=int(old['tile_count']) if confirmed else int(old['v7_count'])
        new_gate='CONFIRMED' if confirmed else ('NO_SUFFICIENT_DETECTIONS' if n==0 else 'NOT_CONFIRMED')
        if (str(old.get('full_presence'))!=full or str(old.get('gate'))!=new_gate or int(old.get('v72_count',0))!=new_count):
            repaired+=1
        cached.at[idx,'full_presence']=full
        cached.at[idx,'crop_yes']=yes
        cached.at[idx,'gate']=new_gate
        cached.at[idx,'v72_count']=new_count
    atomic_csv(cached,GATE_CHECKPOINT)
    print('Reparsed cached Qwen responses:',len(cached),'rows,',repaired,'rows corrected')
else:
    print('Chưa có Qwen checkpoint; bỏ qua sửa cache')


In [ ]:
# 14 — ĐÁNH GIÁ CẢ 1301 CẶP / TRUNG THỰC VỚI CÁC CẶP CHƯA ĐỔI
# Chỉ sau khi đã dự đoán mới truy cập gt_count, pair_type để đo MAE/FPR.
gates=pd.read_csv(GATE_CHECKPOINT)
assert gates.group_id.is_unique
assert set(triggers.group_id).issubset(set(gates.group_id))
all_new=data.copy()
lookup=gates.set_index('group_id')['v72_count'].to_dict()
all_new['v72_count']=all_new.apply(lambda r:int(lookup.get(r.group_id,r.v7_count)),axis=1)
all_new['v7_error']=(all_new.v7_count-all_new.gt_count).abs()
all_new['v72_error']=(all_new.v72_count-all_new.gt_count).abs()
all_new['delta_error']=all_new.v72_error-all_new.v7_error
all_new['changed']=all_new.v7_count.ne(all_new.v72_count)
rows=[]
for kind,sub in [('POSITIVE',all_new[all_new.pair_type=='POSITIVE']),
                 ('HARD_NEGATIVE',all_new[all_new.pair_type=='HARD_NEGATIVE']),
                 ('ALL',all_new)]:
    for col,label in [('v7_count','V7'),('v72_count','V7.2_Exploratory')]:
        err=(sub[col]-sub.gt_count).abs()
        rows.append({'subset':kind,'method':label,'n':len(sub),'MAE':float(err.mean()),
           'RMSE':float(np.sqrt((err**2).mean())),
           'FPR':float((sub[col]>0).mean()) if kind=='HARD_NEGATIVE' else np.nan,
           'positive_zero_rate':float((sub[col]==0).mean()) if kind=='POSITIVE' else np.nan})
metrics=pd.DataFrame(rows)
print(metrics.to_string(index=False))
print('Số cặp thay đổi:',int(all_new.changed.sum()))
print('Tổng chênh lệch tuyệt đối sai số (+ là xấu):',int(all_new.delta_error.sum()))
print('Kích hoạt trên các nhóm (chỉ dùng để phân tích, không dùng chọn ảnh):')
print(triggers.pair_type.value_counts().to_string())

atomic_csv(all_new,OUT/'v72_exploratory_1301_predictions.csv')
atomic_csv(metrics,OUT/'v72_exploratory_metrics.csv')
atomic_csv(all_new[all_new.changed].sort_values('delta_error'),OUT/'v72_changed_cases.csv')
atomic_csv(triggers[['group_id','image_name','target','image_path']],OUT/'v72_trigger_manifest.csv')
print('Exported all results.')


In [ ]:
# 15 — ĐÓNG GÓI CHECKPOINT + BÁO CÁO
import zipfile
from IPython.display import FileLink,display
archive=ROOT/'V7_2_Adaptive_Tiling_Exploratory_1301.zip'
files=[TILE_CHECKPOINT,GATE_CHECKPOINT,
       OUT/'v72_exploratory_1301_predictions.csv',OUT/'v72_exploratory_metrics.csv',
       OUT/'v72_changed_cases.csv',OUT/'v72_trigger_manifest.csv']
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if f.exists():z.write(f,arcname=f.name)
    for f in BOX_STORE.glob('*.json'):
        z.write(f,arcname='tile_boxes/'+f.name)
print('ZIP:',archive,'MB',round(archive.stat().st_size/1024**2,2))
display(FileLink(str(archive)))
